# Chapter 40: Stereo Vision

This notebook builds a compact, reproducible stereo-vision pipeline around the main conceptual arc of MIT *Foundations of Computer Vision* Chapter 40: intuitive stereo cues, binocular geometry, disparity, epipolar constraints, rectification, correspondence, validation, and failure modes.

## Connection to the MIT Vision Book

The conceptual source for this notebook is Chapter 40 of the MIT Vision Book: [Stereo Vision](https://visionbook.mit.edu/3d_scene_understanding_stereo.html). Rather than copying textbook figures, the notebook recreates the core ideas with original code-generated diagrams, synthetic scenes, and quantitative experiments.

The notebook mirrors the chapter's understanding path:

1. Intuition first: what extra information a second view provides.
2. Geometry next: why correspondence is constrained and why disparity implies inverse depth.
3. Computation next: how local block matching turns that geometry into a disparity map.
4. Validation last: when the method works, when it fails, and why.

## Intuition: What information does a second view provide?

This section recreates the intuition behind the book's simple stereo setup figure. A single 3D point projects to different horizontal positions in the left and right cameras, and the horizontal offset becomes a cue for depth.

![Stereo setup](images/01-stereo-setup.png)

## Epipolar Geometry and Rectification

Here we reproduce the book's epipolar-search idea. For arbitrary camera poses, a match in the second view must lie on an epipolar line. After rectification, that search becomes one-dimensional along a scanline.

![Epipolar geometry and rectification](images/02-epipolar-and-rectification.png)

## Why is disparity a proxy for inverse depth?

For a rectified stereo pair with focal length $f$ and baseline $B$, the textbook depth formula is

$$Z = \frac{fB}{d}$$

where $Z$ is depth and $d = x_L - x_R$ is disparity. Nearby points generate large disparities, while faraway points generate small ones.

![Disparity-depth relationship](images/03-disparity-depth-relationship.png)

## Synthetic Rectified Stereo Pair

The experiments below use synthetic data first so ground-truth depth and disparity are known exactly. This makes the validation honest and reproducible, and it keeps the pipeline aligned with the chapter's geometry-first emphasis.

![Synthetic stereo pair](images/04-synthetic-stereo-pair.png)

## Block Matching

The stereo correspondence problem asks: for each left-image patch, where is the matching patch in the right image? In rectified stereo that search stays on the same row. Local block matching works best when there is enough local texture to disambiguate the match.

![Block matching results](images/05-block-matching-results.png)

## Validation, Parameter Tradeoffs, and Failure Cases

This section makes the chapter's caveats computationally explicit. We report mean absolute disparity error and bad-pixel ratio, then show how patch size and search range change the results. Finally, we isolate three classic failure modes: textureless regions, repeated patterns, and occlusion boundaries.

![Parameter sweep and failure cases](images/06-parameter-and-failure-cases.png)

## Limits of This Demonstration

This notebook is deliberately compact. It demonstrates the geometry and mechanics of stereo matching, not a production-quality stereo system. It omits learned cost volumes, left-right consistency checks, subpixel refinement, regularization, and robust photometric modeling.

In [ ]:
# Parameters (overridden by papermill when executed through the repo tooling)
output_dir = "."
images_dir = "./images"

In [ ]:
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

torch.manual_seed(7)
np.random.seed(7)

OUTPUT_DIR = Path(output_dir)
IMAGES_DIR = Path(images_dir)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#fbfbf8",
        "axes.edgecolor": "#3f3f46",
        "axes.grid": True,
        "grid.color": "#d6d3d1",
        "grid.alpha": 0.45,
        "grid.linestyle": "--",
        "font.size": 11,
    }
)

DEVICE = torch.device("cpu")
FOCAL_LENGTH = 84.0
BASELINE = 0.18
HEIGHT = 120
WIDTH = 180
PATCH_SIZE = 7
MAX_DISPARITY = 32

In [ ]:
def depth_to_disparity(depth, focal_length, baseline):
    return focal_length * baseline / torch.clamp(depth, min=1e-6)


def disparity_to_depth(disparity, focal_length, baseline):
    return focal_length * baseline / torch.clamp(disparity, min=1e-6)


def create_synthetic_scene(height=HEIGHT, width=WIDTH, focal_length=FOCAL_LENGTH, baseline=BASELINE):
    y, x = torch.meshgrid(
        torch.arange(height, dtype=torch.float32, device=DEVICE),
        torch.arange(width, dtype=torch.float32, device=DEVICE),
        indexing="ij",
    )

    depth = torch.full((height, width), 8.0, dtype=torch.float32, device=DEVICE)
    left = (
        0.28
        + 0.18 * torch.sin(x / 9.0)
        + 0.13 * torch.cos(y / 11.0)
        + 0.08 * torch.sin((x + 1.4 * y) / 13.0)
        + 0.03 * torch.randn((height, width), device=DEVICE)
    )

    near_box = (x > 18) & (x < 72) & (y > 26) & (y < 90)
    mid_circle = (x - 118) ** 2 + (y - 52) ** 2 < 20**2
    far_ramp = (x > 86) & (x < 160) & (y > 70) & (y < 108)
    occluder = (x > 82) & (x < 95) & (y > 18) & (y < 95)
    repeated = (x > 120) & (x < 172) & (y > 18) & (y < 62)
    textureless = (x > 18) & (x < 68) & (y > 90) & (y < 114)

    depth[y < 28] = 10.0
    depth[near_box] = 2.1
    depth[mid_circle] = 3.5
    depth[far_ramp] = torch.minimum(
        depth[far_ramp],
        3.2 + 2.0 * ((x[far_ramp] - 86.0) / (160.0 - 86.0)),
    )
    depth[occluder] = 1.6
    depth[repeated] = 4.2
    depth[textureless] = 4.7

    left[near_box] = 0.50 + 0.18 * ((((x[near_box] // 6) + (y[near_box] // 6)) % 2)) + 0.05 * torch.sin(
        y[near_box] / 3.0
    )
    left[mid_circle] = 0.22 + 0.55 * torch.exp(-((x[mid_circle] - 118) ** 2 + (y[mid_circle] - 52) ** 2) / 250.0)
    left[far_ramp] = 0.25 + 0.35 * ((x[far_ramp] - 86.0) / (160.0 - 86.0)) + 0.08 * torch.sin(
        y[far_ramp] / 4.0
    )
    left[occluder] = 0.88 - 0.08 * ((((y[occluder] - 18) // 8) % 2))
    left[repeated] = 0.35 + 0.22 * ((((x[repeated] - 120) // 5) % 2)) + 0.05 * torch.cos(
        y[repeated] / 2.0
    )
    left[textureless] = 0.63
    left = torch.clamp(left, 0.0, 1.0)

    disparity = depth_to_disparity(depth, focal_length, baseline)
    right, visible_mask, occlusion_mask = create_rectified_stereo_pair(left, depth, disparity)
    valid_mask = (disparity > 0) & visible_mask

    region_map = {
        "textureless": textureless,
        "repeated_pattern": repeated,
        "occlusion_boundary": occlusion_mask,
    }
    return {
        "left": left,
        "right": right,
        "depth": depth,
        "disparity": disparity,
        "valid_mask": valid_mask,
        "visible_mask": visible_mask,
        "occlusion_mask": occlusion_mask,
        "region_map": region_map,
    }


def create_rectified_stereo_pair(left, depth, disparity):
    height, width = left.shape
    right = torch.full_like(left, float("nan"))
    z_buffer = torch.full_like(left, float("inf"))
    visible_mask = torch.zeros_like(left, dtype=torch.bool)
    occlusion_mask = torch.zeros_like(left, dtype=torch.bool)

    for yy in range(height):
        for xx in range(width):
            disp = int(torch.round(disparity[yy, xx]).item())
            xr = xx - disp
            if xr < 0 or xr >= width:
                occlusion_mask[yy, xx] = True
                continue
            if depth[yy, xx] < z_buffer[yy, xr]:
                right[yy, xr] = left[yy, xx]
                z_buffer[yy, xr] = depth[yy, xx]
                visible_mask[yy, xx] = True

    right_np = right.cpu().numpy()
    for yy in range(height):
        row = right_np[yy]
        finite = np.isfinite(row)
        if finite.any():
            idx = np.where(finite)[0]
            right_np[yy] = np.interp(np.arange(width), idx, row[idx])
        else:
            right_np[yy] = 0.0
    right = torch.from_numpy(right_np).to(DEVICE, dtype=torch.float32)
    return torch.clamp(right, 0.0, 1.0), visible_mask, occlusion_mask


def extract_patch(image, center, patch_size):
    radius = patch_size // 2
    yy, xx = center
    return image[yy - radius : yy + radius + 1, xx - radius : xx + radius + 1]


def compute_ssd(patch1, patch2):
    diff = patch1 - patch2
    return torch.sum(diff * diff)


def block_matching_disparity(left, right, patch_size, max_disparity):
    height, width = left.shape
    radius = patch_size // 2
    disparity = torch.zeros_like(left)
    for yy in range(radius, height - radius):
        for xx in range(radius + max_disparity, width - radius):
            left_patch = extract_patch(left, (yy, xx), patch_size)
            best_cost = float("inf")
            best_disp = 0
            for disp in range(max_disparity + 1):
                xr = xx - disp
                if xr - radius < 0:
                    break
                right_patch = extract_patch(right, (yy, xr), patch_size)
                cost = compute_ssd(left_patch, right_patch).item()
                if cost < best_cost:
                    best_cost = cost
                    best_disp = disp
            disparity[yy, xx] = float(best_disp)
    return disparity


def compute_disparity_error(pred, gt, valid_mask):
    return torch.abs(pred - gt) * valid_mask.float()


def compute_bad_pixel_ratio(pred, gt, threshold, valid_mask):
    bad = ((torch.abs(pred - gt) > threshold) & valid_mask).float().sum()
    total = valid_mask.float().sum().clamp(min=1.0)
    return (bad / total).item()

In [ ]:
def save_current_figure(name):
    plt.savefig(IMAGES_DIR / name, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()


scene = create_synthetic_scene()
print(
    "Synthetic scene created:",
    {
        "image_shape": tuple(scene["left"].shape),
        "disparity_range": (
            float(scene["disparity"][scene["valid_mask"]].min().item()),
            float(scene["disparity"][scene["valid_mask"]].max().item()),
        ),
        "depth_range": (
            float(scene["depth"].min().item()),
            float(scene["depth"].max().item()),
        ),
    },
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(plt.imread(IMAGES_DIR / "01-stereo-setup.png"))
ax.axis("off")
plt.title("Recreated stereo setup intuition")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
ax.imshow(plt.imread(IMAGES_DIR / "02-epipolar-and-rectification.png"))
ax.axis("off")
plt.title("Epipolar constraint and why rectification matters")
plt.show()

In [ ]:
disparities = torch.linspace(1.0, 36.0, 300)
depths = disparity_to_depth(disparities, FOCAL_LENGTH, BASELINE)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(disparities.cpu(), depths.cpu(), color="#0f766e", lw=2.7)
axes[0].set_xlabel("Disparity d (pixels)")
axes[0].set_ylabel("Depth Z")
axes[0].set_title("Inverse relationship from the textbook formula")

sample_disp = torch.tensor([4.0, 8.0, 16.0, 28.0])
sample_depth = disparity_to_depth(sample_disp, FOCAL_LENGTH, BASELINE)
axes[1].bar(["far", "mid", "near"], [4.0, 10.0, 22.0], color=["#94a3b8", "#f59e0b", "#0f766e"])
axes[1].set_ylabel("Illustrative disparity")
axes[1].set_title("Why nearby points move more")
save_current_figure("03-disparity-depth-relationship.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
axes[0].imshow(scene["left"].cpu(), cmap="gray", vmin=0.0, vmax=1.0)
axes[0].set_title("Left rectified image")
axes[0].axis("off")

axes[1].imshow(scene["right"].cpu(), cmap="gray", vmin=0.0, vmax=1.0)
axes[1].set_title("Right rectified image")
axes[1].axis("off")
save_current_figure("04-synthetic-stereo-pair.png")

In [ ]:
start = time.perf_counter()
pred_disparity = block_matching_disparity(scene["left"], scene["right"], PATCH_SIZE, MAX_DISPARITY)
runtime_seconds = time.perf_counter() - start

error_map = compute_disparity_error(pred_disparity, scene["disparity"], scene["valid_mask"])
mae = (
    error_map.sum() / scene["valid_mask"].float().sum().clamp(min=1.0)
).item()
bad_1px = compute_bad_pixel_ratio(pred_disparity, scene["disparity"], threshold=1.0, valid_mask=scene["valid_mask"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, image, title, cmap in [
    (axes[0], scene["disparity"], "Ground-truth disparity", "viridis"),
    (axes[1], pred_disparity, f"Estimated disparity\npatch={PATCH_SIZE}, max d={MAX_DISPARITY}", "viridis"),
    (axes[2], error_map, f"Absolute error\nMAE={mae:.2f}px", "magma"),
]:
    im = ax.imshow(image.cpu(), cmap=cmap)
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
save_current_figure("05-block-matching-results.png")

print({"mae": mae, "bad_pixel_ratio_1px": bad_1px, "runtime_seconds": runtime_seconds})

In [ ]:
patch_sizes = [3, 5, 7, 9]
max_disparities = [16, 24, 32, 40]
mae_grid = torch.zeros((len(patch_sizes), len(max_disparities)))
bad_grid = torch.zeros_like(mae_grid)
runtime_grid = torch.zeros_like(mae_grid)

for i, patch_size in enumerate(patch_sizes):
    for j, max_disp in enumerate(max_disparities):
        start = time.perf_counter()
        pred = block_matching_disparity(scene["left"], scene["right"], patch_size, max_disp)
        runtime_grid[i, j] = time.perf_counter() - start
        err = compute_disparity_error(pred, scene["disparity"], scene["valid_mask"])
        mae_grid[i, j] = err.sum() / scene["valid_mask"].float().sum().clamp(min=1.0)
        bad_grid[i, j] = compute_bad_pixel_ratio(pred, scene["disparity"], threshold=1.0, valid_mask=scene["valid_mask"])

texture_mask = scene["region_map"]["textureless"] & scene["valid_mask"]
repeated_mask = scene["region_map"]["repeated_pattern"] & scene["valid_mask"]
boundary_mask = scene["occlusion_mask"] & scene["valid_mask"]
failure_mae = []
for mask in [texture_mask, repeated_mask, boundary_mask]:
    if mask.any():
        failure_mae.append(error_map[mask].mean().item())
    else:
        failure_mae.append(0.0)

fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 3, height_ratios=[1.05, 1.0])

ax_mae = fig.add_subplot(gs[0, 0])
im1 = ax_mae.imshow(mae_grid.cpu(), cmap="viridis")
ax_mae.set_title("Mean absolute disparity error")
ax_mae.set_xticks(range(len(max_disparities)), max_disparities)
ax_mae.set_yticks(range(len(patch_sizes)), patch_sizes)
ax_mae.set_xlabel("Max disparity")
ax_mae.set_ylabel("Patch size")
fig.colorbar(im1, ax=ax_mae, fraction=0.046, pad=0.04)

ax_bad = fig.add_subplot(gs[0, 1])
im2 = ax_bad.imshow(bad_grid.cpu(), cmap="magma")
ax_bad.set_title("Bad-pixel ratio (> 1 px)")
ax_bad.set_xticks(range(len(max_disparities)), max_disparities)
ax_bad.set_yticks(range(len(patch_sizes)), patch_sizes)
ax_bad.set_xlabel("Max disparity")
ax_bad.set_ylabel("Patch size")
fig.colorbar(im2, ax=ax_bad, fraction=0.046, pad=0.04)

ax_bar = fig.add_subplot(gs[0, 2])
ax_bar.bar(["Textureless", "Repeated", "Occlusion"], failure_mae, color=["#94a3b8", "#f59e0b", "#dc2626"])
ax_bar.set_title("Failure-mode MAE")
ax_bar.set_ylabel("MAE (pixels)")

for idx, (y0, x0, h, w, title) in enumerate(
    [
        (92, 18, 22, 48, "Textureless region"),
        (22, 122, 34, 44, "Repeated pattern"),
        (25, 76, 48, 34, "Occlusion boundary"),
    ]
):
    ax = fig.add_subplot(gs[1, idx])
    crop = scene["left"][y0 : y0 + h, x0 : x0 + w].cpu()
    crop_err = error_map[y0 : y0 + h, x0 : x0 + w].cpu()
    ax.imshow(crop, cmap="gray", vmin=0.0, vmax=1.0)
    ax.imshow(crop_err, cmap="inferno", alpha=0.65)
    ax.set_title(title)
    ax.axis("off")

save_current_figure("06-parameter-and-failure-cases.png")

summary = {
    "patch_sizes": patch_sizes,
    "max_disparities": max_disparities,
    "mae_grid": mae_grid.tolist(),
    "bad_grid": bad_grid.tolist(),
    "runtime_grid_seconds": runtime_grid.tolist(),
    "failure_mae": failure_mae,
}
summary

In [ ]:
metrics = {
    "mean_absolute_disparity_error": mae,
    "bad_pixel_ratio_at_1px": bad_1px,
    "depth_mae": (
        torch.abs(
            disparity_to_depth(torch.clamp(pred_disparity, min=1.0), FOCAL_LENGTH, BASELINE)
            - scene["depth"]
        )[scene["valid_mask"]].mean().item()
    ),
    "runtime_seconds": runtime_seconds,
}
metrics